In [1]:
import pandas as pd
import numpy as np

# Import PyTorch - the deep learning library we'll use to build LSTM
import torch
import torch.nn as nn


train_df = pd.read_parquet("../data/processed/train_final.parquet")
val_df = pd.read_parquet("../data/processed/val_final.parquet")

print(train_df.shape, val_df.shape)
print("PyTorch version:", torch.__version__)

(1085255, 73) (230301, 73)
PyTorch version: 2.14.0+cpu


In [2]:
# List of feature columns (same as before, excluding patient_id and target)
exclude_cols = ["patient_id", "SepsisLabel"]
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

print(len(feature_cols), "features")

# Function to convert a dataframe into a list of per-patient sequences
def create_sequences(data, feature_cols, max_len=100):
    sequences = []
    labels = []
    
    # Loop through each unique patient
    for pid in data["patient_id"].unique():
        # Get this patient's data, sorted by time (ICULOS)
        patient_data = data[data["patient_id"] == pid].sort_values("ICULOS")
        
        # Extract just the feature values as a numpy array
        seq = patient_data[feature_cols].values
        
        # Extract the sepsis labels for each hour
        label = patient_data["SepsisLabel"].values
        
        # If patient's stay is longer than max_len, cut it to max_len hours
        if len(seq) > max_len:
            seq = seq[:max_len]
            label = label[:max_len]
        
        sequences.append(seq)
        labels.append(label)
    
    return sequences, labels

print("function ready")

71 features
function ready


In [3]:
# Take only the first 500 patients as a quick test, before running on the full dataset
sample_patients = train_df["patient_id"].unique()[:500]
sample_df = train_df[train_df["patient_id"].isin(sample_patients)]

# Create sequences for this small sample
sample_sequences, sample_labels = create_sequences(sample_df, feature_cols)

# Print how many sequences we got, and the shape of the first one
print("Number of sequences:", len(sample_sequences))
print("First sequence shape:", sample_sequences[0].shape)
print("First sequence's labels shape:", sample_labels[0].shape)

Number of sequences: 500
First sequence shape: (54, 71)
First sequence's labels shape: (54,)


In [4]:
# Create sequences for the FULL training set
train_sequences, train_labels = create_sequences(train_df, feature_cols)
print("Train sequences created:", len(train_sequences))

# Create sequences for the FULL validation set
val_sequences, val_labels = create_sequences(val_df, feature_cols)
print("Val sequences created:", len(val_sequences))

Train sequences created: 28235
Val sequences created: 6050


In [5]:
# Import PyTorch's padding utility
from torch.nn.utils.rnn import pad_sequence

# Function to convert our list of sequences into a padded PyTorch tensor
def prepare_padded_data(sequences, labels):
    # Convert each patient's sequence (numpy array) into a PyTorch tensor
    seq_tensors = [torch.tensor(seq, dtype=torch.float32) for seq in sequences]
    label_tensors = [torch.tensor(lab, dtype=torch.float32) for lab in labels]
    
    # Record the actual (true) length of each sequence before padding
    lengths = [len(seq) for seq in sequences]
    
    # Pad all sequences to the same length (the longest one in this set), filling extra spots with 0
    padded_seqs = pad_sequence(seq_tensors, batch_first=True, padding_value=0)
    padded_labels = pad_sequence(label_tensors, batch_first=True, padding_value=0)
    
    return padded_seqs, padded_labels, lengths

# Apply this to our small sample first, to test it works
sample_padded_seqs, sample_padded_labels, sample_lengths = prepare_padded_data(sample_sequences, sample_labels)

print("Padded shape:", sample_padded_seqs.shape)
print("Example lengths (first 5 patients):", sample_lengths[:5])

Padded shape: torch.Size([500, 100, 71])
Example lengths (first 5 patients): [54, 23, 29, 48, 17]


In [6]:
# Apply padding to the full training set
train_padded_seqs, train_padded_labels, train_lengths = prepare_padded_data(train_sequences, train_labels)
print("Train padded shape:", train_padded_seqs.shape)

# Apply padding to the full validation set
val_padded_seqs, val_padded_labels, val_lengths = prepare_padded_data(val_sequences, val_labels)
print("Val padded shape:", val_padded_seqs.shape)

Train padded shape: torch.Size([28235, 100, 71])
Val padded shape: torch.Size([6050, 100, 71])


In [7]:
# Define our LSTM-based neural network for sepsis prediction
class SepsisLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=64):
        super().__init__()
        
        # LSTM layer: reads the sequence hour-by-hour and learns temporal patterns
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, 
                             batch_first=True)
        
        # Dropout layer: randomly "turns off" some connections during training to prevent overfitting
        self.dropout = nn.Dropout(0.3)
        
        # Final layer: converts LSTM's output into a single risk score per hour
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # Pass the sequence through the LSTM layer
        lstm_out, _ = self.lstm(x)
        
        # Apply dropout for regularization
        lstm_out = self.dropout(lstm_out)
        
        # Convert to a single number per time-step, then squeeze into a probability (0 to 1) using sigmoid
        out = torch.sigmoid(self.fc(lstm_out))
        
        return out.squeeze(-1)

# Create the model, telling it how many features each hour has (71)
model = SepsisLSTM(n_features=71, hidden_size=64)

print(model)

SepsisLSTM(
  (lstm): LSTM(71, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [8]:
# Calculate class weight for handling imbalance (same idea as XGBoost's scale_pos_weight)
pos_weight_value = (train_padded_labels == 0).sum() / (train_padded_labels == 1).sum()
pos_weight = torch.tensor([pos_weight_value])

print("Positive class weight:", pos_weight_value.item())

# Define the loss function - measures how wrong the model's predictions are
# BCEWithLogitsLoss is used for binary classification, and pos_weight handles imbalance
criterion = nn.BCELoss(reduction='none')  #

Positive class weight: 173.3870086669922


In [9]:
# Calculate positive class weight using ONLY the original, non-padded labels
# (train_labels is the list of arrays we made BEFORE padding - these have correct lengths only)
all_real_labels = np.concatenate(train_labels)

real_neg = (all_real_labels == 0).sum()
real_pos = (all_real_labels == 1).sum()

pos_weight_value = real_neg / real_pos
pos_weight = torch.tensor([pos_weight_value])

print("Correct positive class weight:", pos_weight_value)

Correct positive class weight: 64.21048730776357


In [10]:
# Update the loss function setup with the correct positive class weight
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("setup done with correct weight:", pos_weight_value)

setup done with correct weight: 64.21048730776357


In [11]:
# Number of times we'll go through the entire training data
n_epochs = 5

# Batch size - how many patients we process at once (helps with memory and speed)
batch_size = 64

# Get total number of training patients
n_train = train_padded_seqs.shape[0]

# Convert lengths list to a tensor for easy use
train_lengths_tensor = torch.tensor(train_lengths)

for epoch in range(n_epochs):
    model.train()  # put model in "training mode"
    total_loss = 0
    
    # Process data in batches
    for i in range(0, n_train, batch_size):
        # Get one batch of sequences and labels
        batch_seqs = train_padded_seqs[i:i+batch_size]
        batch_labels = train_padded_labels[i:i+batch_size]
        batch_lengths = train_lengths_tensor[i:i+batch_size]
        
        # Create a mask: 1 for real data, 0 for padding (so padding doesn't affect training)
        max_len = batch_seqs.shape[1]
        mask = torch.arange(max_len).unsqueeze(0) < batch_lengths.unsqueeze(1)
        mask = mask.float()
        
        # Reset gradients from previous step
        optimizer.zero_grad()
        
        # Get model's predictions for this batch
        predictions = model(batch_seqs)
        
        # Calculate loss for every point, then zero-out the padded points using the mask
        raw_loss = criterion(predictions, batch_labels)
        masked_loss = (raw_loss * mask).sum() / mask.sum()
        
        # Backpropagation: calculate how to adjust the model to reduce error
        masked_loss.backward()
        
        # Update the model's internal numbers
        optimizer.step()
        
        total_loss += masked_loss.item()
    
    avg_loss = total_loss / (n_train / batch_size)
    print(f"Epoch {epoch+1}/{n_epochs}, Average Loss: {avg_loss:.4f}")

Epoch 1/5, Average Loss: 0.0928
Epoch 2/5, Average Loss: 0.0807
Epoch 3/5, Average Loss: 0.0796
Epoch 4/5, Average Loss: 0.0789
Epoch 5/5, Average Loss: 0.0783


In [12]:
# Put model in evaluation mode (turns off dropout, etc.)
model.eval()

# We don't need gradient calculation during evaluation (saves memory and time)
with torch.no_grad():
    val_predictions = model(val_padded_seqs)

# Import evaluation metrics
from sklearn.metrics import roc_auc_score, average_precision_score

# We need to extract only the REAL (non-padded) predictions and labels for fair evaluation
all_val_preds = []
all_val_labels = []

for i in range(len(val_lengths)):
    real_len = val_lengths[i]
    # Take only the real time-steps (ignore padded ones) for this patient
    all_val_preds.extend(val_predictions[i, :real_len].numpy())
    all_val_labels.extend(val_padded_labels[i, :real_len].numpy())

# Convert to numpy arrays
all_val_preds = np.array(all_val_preds)
all_val_labels = np.array(all_val_labels)

# Calculate AUROC and AUPRC on only the real data points
auroc_lstm = roc_auc_score(all_val_labels, all_val_preds)
auprc_lstm = average_precision_score(all_val_labels, all_val_preds)

print("LSTM AUROC:", auroc_lstm)
print("LSTM AUPRC:", auprc_lstm)

LSTM AUROC: 0.6456807275531641
LSTM AUPRC: 0.03766662100773194


In [13]:
# Import StandardScaler - this will scale all features to have similar range
from sklearn.preprocessing import StandardScaler

# Create scaler and fit it on training data (learn the mean/std from training data only)
scaler = StandardScaler()

# Reshape training data to 2D temporarily to fit the scaler (it expects 2D input)
n_patients_train, max_len, n_feat = train_padded_seqs.shape
train_flat = train_padded_seqs.numpy().reshape(-1, n_feat)
scaler.fit(train_flat)

# Apply scaling to training data, then reshape back to original 3D shape
train_scaled = scaler.transform(train_flat).reshape(n_patients_train, max_len, n_feat)
train_padded_seqs_scaled = torch.tensor(train_scaled, dtype=torch.float32)

# Apply the SAME scaler (fit on train only) to validation data
n_patients_val = val_padded_seqs.shape[0]
val_flat = val_padded_seqs.numpy().reshape(-1, n_feat)
val_scaled = scaler.transform(val_flat).reshape(n_patients_val, max_len, n_feat)
val_padded_seqs_scaled = torch.tensor(val_scaled, dtype=torch.float32)

print("Scaling done")
print("Train scaled shape:", train_padded_seqs_scaled.shape)

Scaling done
Train scaled shape: torch.Size([28235, 100, 71])


In [18]:
# Create a fresh model (so it doesn't remember the previous, poorly-trained weights)
model = SepsisLSTM(n_features=71, hidden_size=64)

# Fresh optimizer for the new model
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Increase epochs from 5 to 15 for better learning
n_epochs = 15
batch_size = 64
n_train = train_padded_seqs_scaled.shape[0]
train_lengths_tensor = torch.tensor(train_lengths)

for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    
    for i in range(0, n_train, batch_size):
        batch_seqs = train_padded_seqs_scaled[i:i+batch_size]
        batch_labels = train_padded_labels[i:i+batch_size]
        batch_lengths = train_lengths_tensor[i:i+batch_size]
        
        max_len = batch_seqs.shape[1]
        mask = torch.arange(max_len).unsqueeze(0) < batch_lengths.unsqueeze(1)
        mask = mask.float()
        
        optimizer.zero_grad()
        predictions = model(batch_seqs)
        raw_loss = criterion(predictions, batch_labels)
        masked_loss = (raw_loss * mask).sum() / mask.sum()
        masked_loss.backward()
        optimizer.step()
        
        total_loss += masked_loss.item()
    
    avg_loss = total_loss / (n_train / batch_size)
    print(f"Epoch {epoch+1}/{n_epochs}, Average Loss: {avg_loss:.4f}")

Epoch 1/15, Average Loss: 0.0929
Epoch 2/15, Average Loss: 0.0743
Epoch 3/15, Average Loss: 0.0727
Epoch 4/15, Average Loss: 0.0718
Epoch 5/15, Average Loss: 0.0711
Epoch 6/15, Average Loss: 0.0708
Epoch 7/15, Average Loss: 0.0704
Epoch 8/15, Average Loss: 0.0701
Epoch 9/15, Average Loss: 0.0698
Epoch 10/15, Average Loss: 0.0695
Epoch 11/15, Average Loss: 0.0693
Epoch 12/15, Average Loss: 0.0690
Epoch 13/15, Average Loss: 0.0687
Epoch 14/15, Average Loss: 0.0684
Epoch 15/15, Average Loss: 0.0681


In [19]:
model.eval()

with torch.no_grad():
    val_predictions = model(val_padded_seqs_scaled)

all_val_preds = []
all_val_labels = []

for i in range(len(val_lengths)):
    real_len = val_lengths[i]
    all_val_preds.extend(val_predictions[i, :real_len].numpy())
    all_val_labels.extend(val_padded_labels[i, :real_len].numpy())

all_val_preds = np.array(all_val_preds)
all_val_labels = np.array(all_val_labels)

auroc_lstm = roc_auc_score(all_val_labels, all_val_preds)
auprc_lstm = average_precision_score(all_val_labels, all_val_preds)

print("LSTM AUROC:", auroc_lstm)
print("LSTM AUPRC:", auprc_lstm)

LSTM AUROC: 0.7767052546268587
LSTM AUPRC: 0.07980110445323213


In [20]:
# Save the trained LSTM model
torch.save(model.state_dict(), "../models/lstm_model.pt")

# Save results for comparison
import json
lstm_results = {
    "model": "LSTM",
    "AUROC": auroc_lstm,
    "AUPRC": auprc_lstm,
    "epochs": 15
}
with open("../reports/lstm_results.json", "w") as f:
    json.dump(lstm_results, f, indent=2)

print("LSTM saved!")

LSTM saved!
